In [1]:
import pandas as pd
df = pd.read_csv('retail_store_sales.csv')
print(df.columns.tolist())
print(df.dtypes)
print(df.shape)
df.head(10)

['Transaction ID', 'Customer ID', 'Category', 'Item', 'Price Per Unit', 'Quantity', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date', 'Discount Applied']
Transaction ID       object
Customer ID          object
Category             object
Item                 object
Price Per Unit      float64
Quantity            float64
Total Spent         float64
Payment Method       object
Location             object
Transaction Date     object
Discount Applied     object
dtype: object
(12575, 11)


,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False
5,TXN_7482416,CUST_09,Patisserie,NaN,NaN,10.0,200.0,Credit Card,Online,2023-11-30,NaN
6,TXN_3652209,CUST_07,Food,Item_1_FOOD,5.0,8.0,40.0,Credit Card,In-store,2023-06-10,True
7,TXN_1372952,CUST_21,Furniture,NaN,33.5,NaN,NaN,Digital Wallet,In-store,2024-04-02,True
8,TXN_9728486,CUST_23,Furniture,Item_16_FUR,27.5,1.0,27.5,Credit Card,In-store,2023-04-26,False
9,TXN_2722661,CUST_25,Butchers,Item_22_BUT,36.5,3.0,109.5,Cash,Online,2024-03-14,False


In [2]:
# Solo si no lo tienes ya instalado (puedes correrlo, no hace daño si ya está)
!pip install psycopg2-binary sqlalchemy

In [9]:
from sqlalchemy import create_engine

usuario = 'postgres'
contraseña = 'av595749lc'
host = 'localhost'
puerto = '5432'
base_datos = 'retail_sales_db'

engine = create_engine(f'postgresql://{usuario}:{contraseña}@{host}:{puerto}/{base_datos}')

# Renombrar columnas del DataFrame para que coincidan con la tabla SQL
df_renamed = df.rename(columns={
    'Transaction ID': 'transaction_id',
    'Customer ID': 'customer_id',
    'Category': 'category',
    'Item': 'item',
    'Price Per Unit': 'price_per_unit',
    'Quantity': 'quantity',
    'Total Spent': 'total_spent',
    'Payment Method': 'payment_method',
    'Location': 'location',
    'Transaction Date': 'transaction_date',
    'Discount Applied': 'discount_applied'
})

# Cargar a la tabla raw_sales ya creada
df_renamed.to_sql('raw_sales', engine, if_exists='append', index=False)

print("Carga completada:", df_renamed.shape[0], "filas")
from sqlalchemy import text

# Vaciar la tabla antes de recargar (evita duplicados por reejecución accidental)
with engine.connect() as conn:
    conn.execute(text("TRUNCATE TABLE raw_sales"))
    conn.commit()

print("Tabla vaciada")
df_renamed.to_sql('raw_sales', engine, if_exists='append', index=False)
print("Carga completada:", df_renamed.shape[0], "filas")

Carga completada: 12575 filas
Tabla vaciada
Carga completada: 12575 filas


In [10]:
import pandas as pd

check = pd.read_sql('SELECT COUNT(*) as total_filas FROM raw_sales', engine)
print(check)

preview = pd.read_sql('SELECT * FROM raw_sales LIMIT 10', engine)
preview

   total_filas
0        12575


,transaction_id,customer_id,category,item,price_per_unit,quantity,total_spent,payment_method,location,transaction_date,discount_applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,true
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,true
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,false
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,None
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,false
5,TXN_7482416,CUST_09,Patisserie,None,NaN,10.0,200.0,Credit Card,Online,2023-11-30,None
6,TXN_3652209,CUST_07,Food,Item_1_FOOD,5.0,8.0,40.0,Credit Card,In-store,2023-06-10,true
7,TXN_1372952,CUST_21,Furniture,None,33.5,NaN,NaN,Digital Wallet,In-store,2024-04-02,true
8,TXN_9728486,CUST_23,Furniture,Item_16_FUR,27.5,1.0,27.5,Credit Card,In-store,2023-04-26,false
9,TXN_2722661,CUST_25,Butchers,Item_22_BUT,36.5,3.0,109.5,Cash,Online,2024-03-14,false


In [12]:
query1 = """
SELECT
    COUNT(*) AS total_filas,
    COUNT(*) - COUNT(transaction_id)   AS nulos_transaction_id,
    COUNT(*) - COUNT(customer_id)      AS nulos_customer_id,
    COUNT(*) - COUNT(category)         AS nulos_category,
    COUNT(*) - COUNT(item)             AS nulos_item,
    COUNT(*) - COUNT(price_per_unit)   AS nulos_price_per_unit,
    COUNT(*) - COUNT(quantity)         AS nulos_quantity,
    COUNT(*) - COUNT(total_spent)      AS nulos_total_spent,
    COUNT(*) - COUNT(payment_method)   AS nulos_payment_method,
    COUNT(*) - COUNT(location)         AS nulos_location,
    COUNT(*) - COUNT(transaction_date) AS nulos_transaction_date,
    COUNT(*) - COUNT(discount_applied) AS nulos_discount_applied
FROM raw_sales;
"""

resultado1 = pd.read_sql(query1, engine)
print(resultado1)

   total_filas  nulos_transaction_id  nulos_customer_id  nulos_category  \
0        12575                     0                  0               0   

   nulos_item  nulos_price_per_unit  nulos_quantity  nulos_total_spent  \
0        1213                   609             604                604   

   nulos_payment_method  nulos_location  nulos_transaction_date  \
0                     0               0                       0   

   nulos_discount_applied  
0                    4199  


In [13]:
# Query 2 — Duplicados en transaction_id
query2 = """
SELECT transaction_id, COUNT(*) AS veces
FROM raw_sales
GROUP BY transaction_id
HAVING COUNT(*) > 1;
"""
resultado2 = pd.read_sql(query2, engine)
print(resultado2)

Empty DataFrame
Columns: [transaction_id, veces]
Index: []


In [15]:
# Query 3a — Filas donde price_per_unit * quantity no coincide con total_spent
query3a = """
SELECT
    transaction_id,
    price_per_unit,
    quantity,
    total_spent,
    ROUND(price_per_unit * quantity, 2) AS total_calculado,
    ROUND(total_spent - (price_per_unit * quantity), 2) AS diferencia
FROM raw_sales
WHERE price_per_unit IS NOT NULL
  AND quantity IS NOT NULL
  AND total_spent IS NOT NULL
  AND ROUND(total_spent - (price_per_unit * quantity), 2) <> 0
LIMIT 20;
"""
resultado3a = pd.read_sql(query3a, engine)
print(resultado3a)

# Query 3b — Conteo total de inconsistencias
query3b = """
SELECT COUNT(*) AS filas_inconsistentes
FROM raw_sales
WHERE price_per_unit IS NOT NULL
  AND quantity IS NOT NULL
  AND total_spent IS NOT NULL
  AND ROUND(total_spent - (price_per_unit * quantity), 2) <> 0;
"""
resultado3b = pd.read_sql(query3b, engine)
print(resultado3b)

Empty DataFrame
Columns: [transaction_id, price_per_unit, quantity, total_spent, total_calculado, diferencia]
Index: []
   filas_inconsistentes
0                     0


In [16]:
# Query 4 — Valores únicos de discount_applied
query4 = """
SELECT discount_applied, COUNT(*) 
FROM raw_sales
GROUP BY discount_applied;
"""
resultado4 = pd.read_sql(query4, engine)
print(resultado4)

  discount_applied  count
0             None   4199
1             true   4219
2            false   4157


In [17]:
# Query 5 — Rangos generales
query5 = """
SELECT
    MIN(transaction_date) AS fecha_min,
    MAX(transaction_date) AS fecha_max,
    MIN(price_per_unit) AS precio_min,
    MAX(price_per_unit) AS precio_max,
    MIN(quantity) AS cantidad_min,
    MAX(quantity) AS cantidad_max,
    MIN(total_spent) AS total_min,
    MAX(total_spent) AS total_max
FROM raw_sales;
"""
resultado5 = pd.read_sql(query5, engine)
print(resultado5)

    fecha_min   fecha_max  precio_min  precio_max  cantidad_min  cantidad_max  \
0  2022-01-01  2025-01-18         5.0        41.0           1.0          10.0   

   total_min  total_max  
0        5.0      410.0  


In [18]:
# Query 6 — Valores únicos de texto (para detectar inconsistencias de formato)
for col in ['category', 'location', 'payment_method']:
    print(f"\n--- {col} ---")
    print(pd.read_sql(f"SELECT DISTINCT {col} FROM raw_sales ORDER BY {col};", engine))


--- category ---
                             category
0                           Beverages
1                            Butchers
2  Computers and electric accessories
3       Electric household essentials
4                                Food
5                           Furniture
6                       Milk Products
7                          Patisserie

--- location ---
   location
0  In-store
1    Online

--- payment_method ---
   payment_method
0            Cash
1     Credit Card
2  Digital Wallet


El hallazgo clave: nulos recuperables matemáticamente

Como confirmamos que price_per_unit × quantity = total_spent siempre se cumple cuando los tres campos existen, eso significa que si falta solo uno de los tres, lo podemos calcular a partir de los otros dos. No es "descartar datos", es "recuperar datos con lógica de negocio".

In [19]:
query7 = """
SELECT
    COUNT(*) FILTER (
        WHERE price_per_unit IS NULL AND quantity IS NOT NULL AND total_spent IS NOT NULL
    ) AS recuperable_price,
    COUNT(*) FILTER (
        WHERE quantity IS NULL AND price_per_unit IS NOT NULL AND total_spent IS NOT NULL
    ) AS recuperable_quantity,
    COUNT(*) FILTER (
        WHERE total_spent IS NULL AND price_per_unit IS NOT NULL AND quantity IS NOT NULL
    ) AS recuperable_total,
    COUNT(*) FILTER (
        WHERE price_per_unit IS NULL AND quantity IS NULL AND total_spent IS NULL
    ) AS los_tres_nulos,
    COUNT(*) FILTER (
        WHERE (price_per_unit IS NULL AND quantity IS NULL)
           OR (price_per_unit IS NULL AND total_spent IS NULL)
           OR (quantity IS NULL AND total_spent IS NULL)
    ) AS dos_o_mas_nulos_no_recuperables
FROM raw_sales;
"""
resultado7 = pd.read_sql(query7, engine)
print(resultado7)

   recuperable_price  recuperable_quantity  recuperable_total  los_tres_nulos  \
0                609                     0                  0               0   

   dos_o_mas_nulos_no_recuperables  
0                              604  


Esta vista va a ser la que conecte a Power BI (no la tabla cruda). Vamos a aplicar estas reglas:

1. Recuperar price_per_unit cuando falte, vía total_spent / quantity.
2. Excluir las 604 filas donde quantity y total_spent están ambos ausentes (no se pueden recuperar ni tiene sentido imputarlos arbitrariamente — mejor excluir y documentar).
3. item nulo (1,213 filas) → lo dejamos como 'Sin especificar' en vez de excluir, porque no afecta los campos numéricos y sí aporta a los totales de venta por categoría.
4. discount_applied → convertir a booleano real, tratando los nulos como FALSE (asumimos que si no se registró explícitamente el descuento, no se aplicó) — decisión de negocio que hay que documentar como supuesto, no un hecho verificado.

In [21]:
from sqlalchemy import text

with engine.connect() as conn:
    conn.execute(text("""
        CREATE OR REPLACE VIEW clean_sales AS
        SELECT
            transaction_id,
            customer_id,
            category,
            COALESCE(item, 'Sin especificar') AS item,
            COALESCE(price_per_unit, ROUND(total_spent / NULLIF(quantity, 0), 2)) AS price_per_unit,
            quantity,
            total_spent,
            payment_method,
            location,
            transaction_date,
            CASE
                WHEN discount_applied = 'true' THEN TRUE
                WHEN discount_applied = 'false' THEN FALSE
                ELSE FALSE
            END AS discount_applied
        FROM raw_sales
        WHERE NOT (quantity IS NULL AND total_spent IS NULL);
    """))
    conn.commit()

print("Vista clean_sales creada")

Vista clean_sales creada


In [22]:
verificacion = pd.read_sql("""
    SELECT
        COUNT(*) AS total_filas,
        COUNT(*) - COUNT(price_per_unit) AS nulos_price,
        COUNT(*) - COUNT(item) AS nulos_item
    FROM clean_sales;
""", engine)
print(verificacion)

preview = pd.read_sql("SELECT * FROM clean_sales LIMIT 10;", engine)
preview

   total_filas  nulos_price  nulos_item
0        11971            0           0


,transaction_id,customer_id,category,item,price_per_unit,quantity,total_spent,payment_method,location,transaction_date,discount_applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,False
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False
5,TXN_7482416,CUST_09,Patisserie,Sin especificar,20.0,10.0,200.0,Credit Card,Online,2023-11-30,False
6,TXN_3652209,CUST_07,Food,Item_1_FOOD,5.0,8.0,40.0,Credit Card,In-store,2023-06-10,True
7,TXN_9728486,CUST_23,Furniture,Item_16_FUR,27.5,1.0,27.5,Credit Card,In-store,2023-04-26,False
8,TXN_2722661,CUST_25,Butchers,Item_22_BUT,36.5,3.0,109.5,Cash,Online,2024-03-14,False
9,TXN_8776416,CUST_22,Butchers,Item_3_BUT,8.0,9.0,72.0,Cash,In-store,2024-12-14,True


## Auditoría y limpieza de datos (SQL)

**Hallazgos de la auditoría:**
- 0 duplicados en `transaction_id`.
- 0 inconsistencias aritméticas: cuando `price_per_unit`, `quantity` y `total_spent` 
  están presentes, `price_per_unit × quantity` siempre coincide con `total_spent`.
- Nulos concentrados en: `item` (1,213 filas, 9.6%), `price_per_unit`/`quantity`/
  `total_spent` (~604-609 filas), `discount_applied` (4,199 filas, 33.4%).
- Patrón identificado: cuando falta `quantity`, también falta `total_spent` en el 
  100% de los casos (nunca aparecen por separado).

**Reglas de limpieza aplicadas (vista `clean_sales`):**
1. `price_per_unit` nulo (609 filas) → recuperado vía `total_spent / quantity`, 
   aprovechando que la relación aritmética es 100% consistente en el resto del dataset.
2. Filas con `quantity` y `total_spent` ambos nulos (604 filas) → excluidas, al no ser 
   recuperables ni tener base para imputación razonable.
3. `item` nulo (1,213 filas) → reemplazado por `'Sin especificar'` en vez de excluir, 
   ya que no impide el análisis de ventas por categoría/monto.
4. `discount_applied` nulo (4,199 filas) → tratado como `FALSE` (supuesto de negocio: 
   ausencia de registro se interpreta como "sin descuento aplicado").

**Resultado:** de 12,575 filas originales, la vista `clean_sales` conserva 11,971 
(95.2%), lista para conectar a Power BI.